## **Setup**

In [ ]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

In [ ]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

## **Imports**

In [ ]:
from Challenge.paths import load_xgboost_cv_folds, XGBOOST_MODELS
from Challenge.utils import load_models

## **Load Data**

In [ ]:
URM_inner, URM_outer, folds_inner = load_xgboost_cv_folds()

## **Train Models**

In [ ]:
from Recommenders.NonPersonalizedRecommender import TopPop
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.SLIM.SLIMElasticNetRecommender import MultiThreadSLIM_SLIMElasticNetRecommender
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_WARP_Cython
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_BPR_Cython
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_SVDpp_Cython

from Recommenders.SLIM.Cython.SLIM_BPR_Cython import SLIM_BPR_Cython
from implicit.cpu.als import AlternatingLeastSquares
from Recommenders.MatrixFactorization.NMFRecommender import NMFRecommender
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask

### **Candidate Generators**

In [ ]:
candidate_mapping = {
    'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
    'ItemKNN_tversky': ItemKNNCFRecommender,
    'UserKNN_asymmetric': UserKNNCFRecommender,
    'RP3beta': RP3betaRecommender,
    'IALS': AlternatingLeastSquares,
    'TopPop': TopPop
}

candidate_cutoff = {
    'SLIMElasticNet': 80,
    'ItemKNN_tversky': 50,
    'UserKNN_asymmetric': 50,
    'RP3beta': 50,
    'IALS': 50,
    'TopPop': 50
}

In [ ]:
folder = os.path.join(XGBOOST_MODELS, "candidates")
inner_folder = os.path.join(folder, "inner")
outer_folder = os.path.join(folder, "outer")
folds_folder = os.path.join(XGBOOST_MODELS, "folds") 

#### **Inner URM**

In [ ]:
# Load models (lazy loading)
models = load_models(
    URM_train=URM_inner,
    mapping=candidate_mapping,
    model_folder=inner_folder
)

# Force loading of models
# Will train them if needed
[name for name, _ in models]

#### **Outer URM**

In [ ]:
# Load models (lazy loading)
models = load_models(
    URM_train=URM_outer,
    mapping=candidate_mapping,
    model_folder=outer_folder
)

# Force loading of models
# Will train them if needed
[name for name, _ in models]

#### **Folds**

In [ ]:
for i, (URM_train, URM_val) in enumerate(folds_inner):
    print(f"FOLD {i+1}/{len(folds_inner)}")
    
    # Load models (lazy loading)
    models = load_models(
        URM_train=URM_train,
        mapping=candidate_mapping,
        model_folder=os.path.join(folds_folder, f"fold_{i}")
    )
    
    # Force loading of models
    # Will train them if needed
    [name for name, _ in models]

### **Features Generators**

In [ ]:
models_mapping = {
    'TopPop': TopPop,
    'ItemKNN_cosine': ItemKNNCFRecommender,
    'ItemKNN_jaccard': ItemKNNCFRecommender,
    'ItemKNN_asymmetric': ItemKNNCFRecommender,
    'ItemKNN_tversky': ItemKNNCFRecommender,
    'ItemKNN_dice': ItemKNNCFRecommender,
    'UserKNN_cosine': UserKNNCFRecommender,
    'UserKNN_jaccard': UserKNNCFRecommender,
    'UserKNN_asymmetric': UserKNNCFRecommender,
    'UserKNN_tversky': UserKNNCFRecommender,
    'UserKNN_dice': UserKNNCFRecommender,
    'EASE_R': EASE_R_Recommender,
    'P3alpha': P3alphaRecommender,
    'RP3beta': RP3betaRecommender,
    'IALS': AlternatingLeastSquares,
    'MatrixFactorization_WARP': MatrixFactorization_WARP_Cython,
    'MatrixFactorization_BPR': MatrixFactorization_BPR_Cython,
    'MatrixFactorization_SVDpp': MatrixFactorization_SVDpp_Cython,
    
    'SLIM_BPR': SLIM_BPR_Cython,
    'NMF': NMFRecommender,
    'MultVAE': MultVAERecommender_PyTorch_OptimizerMask,
    'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender
}

In [ ]:
folder = os.path.join(XGBOOST_MODELS, "train_features")
inner_folder = os.path.join(folder, "inner")
outer_folder = os.path.join(folder, "outer")
folds_folder = os.path.join(folder, "folds") 

#### **Inner URM**

In [ ]:
# Load models (lazy loading)
models = load_models(
    URM_train=URM_inner,
    mapping=models_mapping,
    model_folder=inner_folder
)

# Force loading of models
# Will train them if needed
[name for name, _ in models]

#### **Inner + Outer URM**

In [ ]:
# Load models (lazy loading)
models = load_models(
    URM_train=URM_inner+URM_outer,
    mapping=models_mapping,
    model_folder=outer_folder
)

# Force loading of models
# Will train them if needed
[name for name, _ in models]

#### **Folds**

In [ ]:
for i, (URM_train, URM_val) in enumerate(folds_inner):
    print(f"FOLD {i+1}/{len(folds_inner)}")
    
    # Load models (lazy loading)
    models = load_models(
        URM_train=URM_train,
        mapping=models_mapping,
        model_folder=os.path.join(folds_folder, f"fold_{i}")
    )
    
    # Force loading of models
    # Will train them if needed
    [name for name, _ in models]